<a href="https://colab.research.google.com/github/jennifer-algabre/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/jennifer-algabre/flyrank-ml-internship.git
%cd flyrank-ml-internship

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.
/content/flyrank-ml-internship


In [2]:
!find . -name "content_refresh_anonymized.csv"

./data/raw/content_refresh_anonymized.csv


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


### Finding 1: Content performance patterns can be identified from search and engagement signals

The research paper presents findings based on observed relationships between content characteristics and search performance outcomes.

**Methodology question:**  
How are the labels or outcomes defined in the analysis? A reviewer should verify whether the labels represent directly observed outcomes or are created from existing rules and thresholds. Clear label definitions are important because the model or analysis can only be as reliable as the target it is measuring.

---

### Finding 2: Data-driven approaches can help identify opportunities for content improvement

The research paper discusses patterns that may help guide content decisions.

**Methodology question:**  
Does the validation approach support the strength of the claim being made? A reviewer should examine whether the evaluation design considers differences between clients, time periods, or other sources of variation. A strong validation strategy helps determine whether observed patterns are likely to generalize beyond the evaluated dataset.

---

These questions are intended as constructive methodology checks. The goal is to understand how the data, labels, and validation design support the conclusions.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*



The Week 5 model used a standard stratified train-test split to preserve the class distribution between training and testing data.

For this audit, the validation approach is improved by using a grouped split based on the pseudonymized client ID. This reduces the possibility that similar patterns from the same client appear in both training and testing sets.

The comparison below shows how model performance changes when evaluated under a stricter validation design.

The results should be interpreted as measured performance on the available dataset and not as proof of future production performance.

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, f1_score


df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create target label
df["label"] = (df["trend_direction"] == "down").astype(int)


features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "search_volume",
    "competition"
]


# Treat avg_position = 0 as missing
df["avg_position"] = df["avg_position"].replace(0, np.nan)


X = df[features].copy()


# Median imputation
for col in X.columns:
    X[col] = X[col].fillna(X[col].median())


y = df["label"]

groups = df["client_id"]


print("Rows:", len(df))
print("Features:", X.shape[1])

Rows: 30000
Features: 11


In [4]:
# Group split by client_id
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)


train_idx, test_idx = next(
    gss.split(X, y, groups)
)


X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]


print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print(
    "Training clients:",
    groups.iloc[train_idx].nunique()
)

print(
    "Testing clients:",
    groups.iloc[test_idx].nunique()
)

Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7


In [5]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)


model.fit(
    X_train,
    y_train
)


pred = model.predict(X_test)


group_accuracy = accuracy_score(
    y_test,
    pred
)

group_precision = precision_score(
    y_test,
    pred
)

group_f1 = f1_score(
    y_test,
    pred
)


print("Grouped Accuracy:", round(group_accuracy, 4))
print("Grouped Precision:", round(group_precision, 4))
print("Grouped F1:", round(group_f1, 4))

Grouped Accuracy: 0.5647
Grouped Precision: 0.5693
Grouped F1: 0.5878


In [6]:
comparison = pd.DataFrame({

    "Validation Method": [
        "Week 5 Stratified Split",
        "Week 6 Grouped Split"
    ],

    "Accuracy": [
        0.6905,
        group_accuracy
    ],

    "Precision": [
        0.701648,
        group_precision
    ],

    "F1 Score": [
        None,
        group_f1
    ]

})


comparison

,Validation Method,Accuracy,Precision,F1 Score
0,Week 5 Stratified Split,0.69050,0.701648,NaN
1,Week 6 Grouped Split,0.56466,0.569345,0.587802


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The final feature set was reviewed for possible data leakage.

The target label was created from `trend_direction`, therefore this field was not included as a model feature. Related outcome-based fields such as `trend_pct` were also excluded because they directly represent the prediction target.

Identifier fields such as `content_id` and `client_id` were not used as model inputs. These fields are only appropriate for grouping, joining, or validation purposes.

The final feature set uses observable search performance and content metadata signals. Product decision flags and target-derived metrics were excluded to ensure the model learns from available evidence rather than from existing decisions.

In [7]:
# Check final feature list for risky columns

leakage_check = {
    "trend_direction_in_features": "trend_direction" in features,
    "trend_pct_in_features": "trend_pct" in features,
    "content_id_in_features": "content_id" in features,
    "client_id_in_features": "client_id" in features
}


pd.DataFrame(
    leakage_check.items(),
    columns=["Check", "Present"]
)

,Check,Present
0,trend_direction_in_features,False
1,trend_pct_in_features,False
2,content_id_in_features,False
3,client_id_in_features,False


In [8]:
importance = pd.DataFrame({

    "Feature": X.columns,

    "Importance": model.feature_importances_

}).sort_values(
    "Importance",
    ascending=False
)


importance

,Feature,Importance
0,impressions_90d,0.206137
3,avg_position,0.159187
7,content_age_days,0.128300
6,word_count,0.114013
5,scroll_rate,0.081668
2,ctr,0.074397
1,clicks_90d,0.059408
9,search_volume,0.051425
10,competition,0.049439
4,engagement_rate,0.040399


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

The Random Forest model performs better than the rule-based baseline because it learns patterns from multiple search performance features instead of relying on a single manually defined rule.

### Revised claim

The Random Forest model showed higher measured predictive performance than the Week 4 baseline on the evaluated dataset. The model combines multiple observable search performance signals to provide decision support for identifying pages that may require review.

These results describe observed relationships in the dataset and should not be interpreted as evidence that any individual feature causes changes in content performance.
Self-check

In [9]:
errors = X_test.copy()

errors["actual"] = y_test
errors["predicted"] = pred


wrong_predictions = errors[
    errors["actual"] != errors["predicted"]
]


print(
    "Number of incorrect predictions:",
    len(wrong_predictions)
)


wrong_predictions.head()

Number of incorrect predictions: 2683


,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,scroll_rate,word_count,content_age_days,days_since_last_update,search_volume,competition,actual,predicted
0,3803,29,0.76,10.6,5.88,4.55,3221.0,187,20,10.0,0.67,1,0
1,15320,7,0.05,20.3,0.00,10.00,2481.0,445,25,90.0,0.01,1,0
13,307,0,0.00,39.8,0.00,25.00,1342.0,238,103,10.0,0.00,0,1
23,297,1,0.34,13.9,0.00,25.00,2877.0,502,20,0.0,0.00,1,0
25,27,0,0.00,7.2,0.00,0.00,2777.0,180,20,70.0,0.81,1,0
